<a href="https://colab.research.google.com/github/JSJeong-me/Hair/blob/main/01-Vector-Similarity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install chromadb

In [ ]:
# prompt: 사전학습 ResNet‑50 불러오기

import tensorflow as tf
from tensorflow.keras.applications.resnet50 import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input, decode_predictions
import numpy as np
from PIL import Image

# 사전 학습된 ResNet-50 모델 불러오기 (ImageNet 가중치 사용)
model = ResNet50(weights='imagenet')

print("사전 학습된 ResNet-50 모델이 성공적으로 로드되었습니다.")


In [ ]:
# prompt: model summary 를 보여 주세요

model.summary()

In [ ]:
# prompt: 'predictions (Dense)' layer 에서 출력 되는 softmax 값으로 vector database 에 embedding 과정을 단계별로 설명과 code 작성

from tensorflow.keras.models import Model

# Get the output tensor of the 'predictions' layer
prediction_layer_output = model.get_layer('predictions').output

# Create a new model that outputs the predictions layer output
prediction_model = Model(inputs=model.input, outputs=prediction_layer_output)

print("Prediction model created successfully.")


In [ ]:
import requests

image_url = 'https://raw.githubusercontent.com/JSJeong-me/GPT-Web/main/images/elephant.jpg'
output_path = '/content/elephant1.jpg'

response = requests.get(image_url)
response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)

with open(output_path, 'wb') as f:
    f.write(response.content)

print(f"Image downloaded and saved to {output_path}")

In [ ]:
import requests

image_url = 'https://raw.githubusercontent.com/JSJeong-me/GPT-Web/main/images/cat1.png'
output_path = '/content/cat1.png'

response = requests.get(image_url)
response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)

with open(output_path, 'wb') as f:
    f.write(response.content)

print(f"Image downloaded and saved to {output_path}")

In [ ]:
# prompt: '/content/elephant1.jpg' 영상을 입력 받아 softmax 값을 출력

# 이미지를 불러와 ResNet-50 입력 크기로 조정
img_path = '/content/elephant1.jpg'  # https://github.com/JSJeong-me/GPT-Web/blob/main/images/elephant.jpg
img = Image.open(img_path).resize((224, 224))

# 이미지를 numpy 배열로 변환하고 ResNet-50 입력 형식에 맞게 전처리
img_array = np.array(img)
img_array = np.expand_dims(img_array, axis=0)
img_array = preprocess_input(img_array)

# Prediction model을 사용하여 예측 결과 (softmax 값) 얻기
predictions = prediction_model.predict(img_array)

# 예측 결과를 출력 (softmax 값)
print("Predicted softmax values:")
print(predictions)

# 예측 결과를 해석하여 상위 N개의 클래스 출력 (선택 사항)
decoded_predictions = decode_predictions(predictions, top=5)[0]
print("\nTop 5 predictions:")
for (imagenet_id, label, score) in decoded_predictions:
    print(f"{label}: {score:.2f}%")


In [ ]:
# ----- [1] 기존 ResNet-50 softmax 예측 코드 이후 -----
import chromadb
from chromadb.utils import embedding_functions
import uuid
import numpy as np

# 예시: predictions 변수는 shape (1, 1000) softmax 결과
embedding_vector = predictions[0].tolist()  # 리스트로 변환

# ----- [2] Chroma DB 컬렉션 생성 -----
client = chromadb.Client()
collection = client.create_collection(name="resnet50_softmax_db")

# ----- [3] 벡터 임베딩 저장 -----
# 각 이미지마다 고유 ID 생성 (예: UUID)
image_id = str(uuid.uuid4())
metadata = {"img_path": img_path}

collection.add(
    embeddings=[embedding_vector],
    metadatas=[metadata],
    ids=[image_id],
)

print(f"Saved embedding to ChromaDB: {image_id}")

In [ ]:
# ----- [4] 검색 (유사 이미지 softmax vector 검색) -----
# 예시로, 같은 이미지를 질의 벡터로 사용 (실제엔 다른 이미지로 검색)
query_vector = predictions[0].tolist()  # shape (1000,)

results = collection.query(
    query_embeddings=[query_vector],
    n_results=3,
    include=["metadatas", "distances"],  # "ids" 제거
)

print("Top 3 similar embeddings:")
for idx, (item_id, meta, dist) in enumerate(zip(results["ids"][0], results["metadatas"][0], results["distances"][0])):
    print(f"{idx+1}. ID: {item_id}, Path: {meta['img_path']}, Distance: {dist}")



In [ ]:
import requests

image_url = 'https://raw.githubusercontent.com/JSJeong-me/GPT-Web/main/images/cat2.jpg'
output_path = '/content/cat2.jpg'

response = requests.get(image_url)
response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)

with open(output_path, 'wb') as f:
    f.write(response.content)

print(f"Image downloaded and saved to {output_path}")

In [ ]:
from PIL import Image
import numpy as np
import uuid

# (1) cat.jpg 이미지 불러오기 및 전처리
cat_img_path = '/content/cat2.jpg' # https://github.com/JSJeong-me/GPT-Web/blob/main/images/cat1.png
cat_img = Image.open(cat_img_path).resize((224, 224))
cat_img_array = np.array(cat_img)
cat_img_array = np.expand_dims(cat_img_array, axis=0)
cat_img_array = preprocess_input(cat_img_array)

# (2) ResNet-50 softmax 임베딩 추출
cat_predictions = prediction_model.predict(cat_img_array)
cat_embedding_vector = cat_predictions[0].tolist()

# (3) Chroma vector DB에 임베딩 추가 저장
cat_image_id = str(uuid.uuid4())
cat_metadata = {"img_path": cat_img_path}

collection.add(
    embeddings=[cat_embedding_vector],
    metadatas=[cat_metadata],
    ids=[cat_image_id],
)

print(f"'cat.jpg' embedding 저장 완료! ID: {cat_image_id}")


ChromaDB에 저장된 'cat'과 'elephant' 이미지의 1000차원 임베딩을 불러와
2차원(PCA 또는 t-SNE 등)으로 차원 축소 후 matplotlib에 시각화하는 예시입니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA  # 또는 sklearn.manifold.TSNE 사용 가능

# ----- 1. ChromaDB에서 모든 임베딩과 메타데이터 불러오기 -----
# ChromaDB에서 모든 벡터(임베딩)와 메타데이터 추출
all_results = collection.get(
    include=['embeddings', 'metadatas']  # "ids"를 빼고 사용!
)
embeddings = np.array(all_results['embeddings'])
metadatas = all_results['metadatas']
ids = all_results['ids']  # 이 줄은 그대로 사용


# ----- 2. 'cat'과 'elephant' 이미지의 인덱스 추출 -----
labels = []
for meta in metadatas:
    # 파일명에 'cat' 또는 'elephant'가 포함된 경우 해당 라벨 부여
    if 'cat' in meta['img_path']:
        labels.append('cat')
    elif 'elephant' in meta['img_path']:
        labels.append('elephant')
    else:
        labels.append('other')

# 'cat' 또는 'elephant'만 추출
selected_idx = [i for i, l in enumerate(labels) if l in ['cat', 'elephant']]
selected_embeddings = embeddings[selected_idx]
selected_labels = [labels[i] for i in selected_idx]

# ----- 3. PCA로 2차원 차원 축소 -----
pca = PCA(n_components=2)
embeddings_2d = pca.fit_transform(selected_embeddings)

# ----- 4. 2D 시각화 -----
plt.figure(figsize=(6, 6))
for label in set(selected_labels):
    idx = [i for i, l in enumerate(selected_labels) if l == label]
    plt.scatter(embeddings_2d[idx, 0], embeddings_2d[idx, 1], label=label, s=100)

plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('2D PCA Embedding: Cat vs Elephant')
plt.legend()
plt.grid(True)
plt.show()
